A Network Intrusion Detection System (NIDS) monitors network traffic and tries to detect:

Unauthorized access

Malware activity

Denial of Service (DoS) attacks

Brute-force attempts

Data exfiltration

Traditional systems use rule-based detection (signatures).
You will build an ML-based anomaly or attack classifier.

Problem Formulation (ML View)

You are given:

Network traffic records (flows)

Each record contains features like:

Source IP

Destination IP

Protocol

Packet length

Duration

Number of bytes

Flags

etc.

And your task is:

Classify each traffic flow as Normal or Attack
OR
Multi-class classification (DoS, Probe, R2L, U2R, etc.)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
data = pd.read_csv("/content/drive/MyDrive/IDS 2018 Intrusion CSV's/02-14-2018.csv")

In [ ]:

data.head()

,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,0,0,14/02/2018 08:31:01,112641719,3,0,0,0,0,0,...,0,0.0,0.0,0,0,56320859.5,139.300036,56320958,56320761,Benign
1,0,0,14/02/2018 08:33:50,112641466,3,0,0,0,0,0,...,0,0.0,0.0,0,0,56320733.0,114.551299,56320814,56320652,Benign
2,0,0,14/02/2018 08:36:39,112638623,3,0,0,0,0,0,...,0,0.0,0.0,0,0,56319311.5,301.934596,56319525,56319098,Benign
3,22,6,14/02/2018 08:40:13,6453966,15,10,1239,2273,744,0,...,32,0.0,0.0,0,0,0.0,0.000000,0,0,Benign
4,22,6,14/02/2018 08:40:23,8804066,14,11,1143,2209,744,0,...,32,0.0,0.0,0,0,0.0,0.000000,0,0,Benign


In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 80 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   Dst Port           1048575 non-null  int64  
 1   Protocol           1048575 non-null  int64  
 2   Timestamp          1048575 non-null  object 
 3   Flow Duration      1048575 non-null  int64  
 4   Tot Fwd Pkts       1048575 non-null  int64  
 5   Tot Bwd Pkts       1048575 non-null  int64  
 6   TotLen Fwd Pkts    1048575 non-null  int64  
 7   TotLen Bwd Pkts    1048575 non-null  int64  
 8   Fwd Pkt Len Max    1048575 non-null  int64  
 9   Fwd Pkt Len Min    1048575 non-null  int64  
 10  Fwd Pkt Len Mean   1048575 non-null  float64
 11  Fwd Pkt Len Std    1048575 non-null  float64
 12  Bwd Pkt Len Max    1048575 non-null  int64  
 13  Bwd Pkt Len Min    1048575 non-null  int64  
 14  Bwd Pkt Len Mean   1048575 non-null  float64
 15  Bwd Pkt Len Std    1048575 non-n

In [ ]:
data.isnull().sum()

,0
Dst Port,0
Protocol,0
Timestamp,0
Flow Duration,0
Tot Fwd Pkts,0
...,...
Idle Mean,0
Idle Std,0
Idle Max,0
Idle Min,0


In [ ]:
data.describe()

,Dst Port,Protocol,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,Fwd Pkt Len Mean,...,Fwd Act Data Pkts,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
count,1.048575e+06,1.048575e+06,1.048575e+06,1.048575e+06,1.048575e+06,1.048575e+06,1.048575e+06,1.048575e+06,1.048575e+06,1.048575e+06,...,1.048575e+06,1.048575e+06,1.048575e+06,1.048575e+06,1.048575e+06,1.048575e+06,1.048575e+06,1.048575e+06,1.048575e+06,1.048575e+06
mean,4.876262e+03,8.107557e+00,6.255555e+06,6.206622e+00,7.211191e+00,4.479936e+02,4.521803e+03,1.745736e+02,8.389535e+00,3.879579e+01,...,2.793536e+00,2.327970e+01,5.152449e+04,2.136151e+04,8.789157e+04,3.995477e+04,3.101206e+06,7.297218e+05,4.812391e+06,2.126920e+06
std,1.444344e+04,4.460625e+00,1.260291e+09,4.447851e+01,1.048682e+02,1.573541e+04,1.515021e+05,2.876713e+02,1.948279e+01,5.331882e+01,...,5.557106e+00,1.106185e+01,5.815586e+05,2.186405e+05,7.395725e+05,5.602693e+05,5.414780e+08,3.820031e+08,1.522117e+09,1.817013e+07
min,0.000000e+00,0.000000e+00,-9.190110e+11,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,2.200000e+01,6.000000e+00,7.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,2.000000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,5.300000e+01,6.000000e+00,1.023000e+03,2.000000e+00,1.000000e+00,3.600000e+01,5.500000e+01,3.400000e+01,0.000000e+00,2.566667e+01,...,0.000000e+00,2.000000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,4.430000e+02,6.000000e+00,4.066690e+05,7.000000e+00,6.000000e+00,4.550000e+02,7.680000e+02,1.990000e+02,0.000000e+00,5.550000e+01,...,4.000000e+00,3.200000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
max,6.553300e+04,1.700000e+01,1.200000e+08,5.115000e+03,9.198000e+03,8.591554e+06,1.339773e+07,6.444000e+04,1.460000e+03,1.121703e+04,...,1.031000e+03,4.800000e+01,1.102401e+08,5.723446e+07,1.102401e+08,1.102401e+08,3.394503e+11,2.432682e+11,9.797810e+11,1.260300e+10


In [ ]:
data["Label"].value_counts()

,count
Label,
Benign,667626
FTP-BruteForce,193360
SSH-Bruteforce,187589


In [ ]:
pd.set_option('display.max_rows', None)
# pd.reset_option('display.max_rows')

from numpy._core import numeric
numeric_data = data.select_dtypes(include = [np.number])

print("null values in the data are : \n")
print(numeric_data.isnull().sum())

print("infinite values in the data are : \n")
print(np.isinf(numeric_data).sum())

null values in the data are : 

Dst Port                0
Protocol                0
Flow Duration           0
Tot Fwd Pkts            0
Tot Bwd Pkts            0
TotLen Fwd Pkts         0
TotLen Bwd Pkts         0
Fwd Pkt Len Max         0
Fwd Pkt Len Min         0
Fwd Pkt Len Mean        0
Fwd Pkt Len Std         0
Bwd Pkt Len Max         0
Bwd Pkt Len Min         0
Bwd Pkt Len Mean        0
Bwd Pkt Len Std         0
Flow Byts/s          2277
Flow Pkts/s             0
Flow IAT Mean           0
Flow IAT Std            0
Flow IAT Max            0
Flow IAT Min            0
Fwd IAT Tot             0
Fwd IAT Mean            0
Fwd IAT Std             0
Fwd IAT Max             0
Fwd IAT Min             0
Bwd IAT Tot             0
Bwd IAT Mean            0
Bwd IAT Std             0
Bwd IAT Max             0
Bwd IAT Min             0
Fwd PSH Flags           0
Bwd PSH Flags           0
Fwd URG Flags           0
Bwd URG Flags           0
Fwd Header Len          0
Bwd Header Len          0
Fwd Pk

In [ ]:
data["Label"].unique()

array(['Benign', 'FTP-BruteForce', 'SSH-Bruteforce'], dtype=object)

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.model_selection import train_test_split

In [ ]:
train_strat,test_strat = train_test_split(
    data,
    test_size = 0.2,
    random_state = 42,
    stratify = data["Label"]
)

In [ ]:
train_main = train_strat

In [ ]:
train_main.head()

,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
135924,21,6,14/02/2018 11:41:38,5,1,1,0,0,0,0,...,40,0.0,0.0,0,0,0.0,0.0,0,0,FTP-BruteForce
571728,53,17,14/02/2018 01:28:51,492,1,1,46,174,46,46,...,8,0.0,0.0,0,0,0.0,0.0,0,0,Benign
1026967,3389,6,14/02/2018 11:25:34,3467731,10,7,1148,1581,677,0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
14149,21,6,14/02/2018 10:40:30,2,1,1,0,0,0,0,...,40,0.0,0.0,0,0,0.0,0.0,0,0,FTP-BruteForce
418098,53,17,14/02/2018 11:10:29,581,1,1,35,51,35,35,...,8,0.0,0.0,0,0,0.0,0.0,0,0,Benign


In [ ]:
train_main.info()

<class 'pandas.core.frame.DataFrame'>
Index: 838860 entries, 135924 to 933846
Data columns (total 80 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Dst Port           838860 non-null  int64  
 1   Protocol           838860 non-null  int64  
 2   Timestamp          838860 non-null  object 
 3   Flow Duration      838860 non-null  int64  
 4   Tot Fwd Pkts       838860 non-null  int64  
 5   Tot Bwd Pkts       838860 non-null  int64  
 6   TotLen Fwd Pkts    838860 non-null  int64  
 7   TotLen Bwd Pkts    838860 non-null  int64  
 8   Fwd Pkt Len Max    838860 non-null  int64  
 9   Fwd Pkt Len Min    838860 non-null  int64  
 10  Fwd Pkt Len Mean   838860 non-null  float64
 11  Fwd Pkt Len Std    838860 non-null  float64
 12  Bwd Pkt Len Max    838860 non-null  int64  
 13  Bwd Pkt Len Min    838860 non-null  int64  
 14  Bwd Pkt Len Mean   838860 non-null  float64
 15  Bwd Pkt Len Std    838860 non-null  float64
 16  Fl

In [ ]:
train_main.isnull().sum()

,0
Dst Port,0
Protocol,0
Timestamp,0
Flow Duration,0
Tot Fwd Pkts,0
Tot Bwd Pkts,0
TotLen Fwd Pkts,0
TotLen Bwd Pkts,0
Fwd Pkt Len Max,0
Fwd Pkt Len Min,0


In [ ]:
train_main["Flow Byts/s"] = train_main["Flow Byts/s"].fillna(train_main["Flow Byts/s"].mean())
train_main.isnull().sum()

,0
Dst Port,0
Protocol,0
Timestamp,0
Flow Duration,0
Tot Fwd Pkts,0
Tot Bwd Pkts,0
TotLen Fwd Pkts,0
TotLen Bwd Pkts,0
Fwd Pkt Len Max,0
Fwd Pkt Len Min,0


In [ ]:
num_data = train_main.select_dtypes("number")
inf = np.isinf(num_data).sum()
inf

,0
Dst Port,0
Protocol,0
Flow Duration,0
Tot Fwd Pkts,0
Tot Bwd Pkts,0
TotLen Fwd Pkts,0
TotLen Bwd Pkts,0
Fwd Pkt Len Max,0
Fwd Pkt Len Min,0
Fwd Pkt Len Mean,0


In [ ]:
fill_inf_null = train_main.replace([np.inf,-np.inf],np.nan,inplace = True)
train_main.isnull().sum()

,0
Dst Port,0
Protocol,0
Timestamp,0
Flow Duration,0
Tot Fwd Pkts,0
Tot Bwd Pkts,0
TotLen Fwd Pkts,0
TotLen Bwd Pkts,0
Fwd Pkt Len Max,0
Fwd Pkt Len Min,0


In [ ]:
train_main["Flow Byts/s"] = train_main["Flow Byts/s"].fillna(train_main["Flow Byts/s"].mean())
train_main["Flow Pkts/s"] = train_main["Flow Pkts/s"].fillna(train_main["Flow Pkts/s"].mean())
train_main.isnull().sum()

,0
Dst Port,0
Protocol,0
Timestamp,0
Flow Duration,0
Tot Fwd Pkts,0
Tot Bwd Pkts,0
TotLen Fwd Pkts,0
TotLen Bwd Pkts,0
Fwd Pkt Len Max,0
Fwd Pkt Len Min,0


In [ ]:
train_main.info()

<class 'pandas.core.frame.DataFrame'>
Index: 838860 entries, 135924 to 933846
Data columns (total 80 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Dst Port           838860 non-null  int64  
 1   Protocol           838860 non-null  int64  
 2   Timestamp          838860 non-null  object 
 3   Flow Duration      838860 non-null  int64  
 4   Tot Fwd Pkts       838860 non-null  int64  
 5   Tot Bwd Pkts       838860 non-null  int64  
 6   TotLen Fwd Pkts    838860 non-null  int64  
 7   TotLen Bwd Pkts    838860 non-null  int64  
 8   Fwd Pkt Len Max    838860 non-null  int64  
 9   Fwd Pkt Len Min    838860 non-null  int64  
 10  Fwd Pkt Len Mean   838860 non-null  float64
 11  Fwd Pkt Len Std    838860 non-null  float64
 12  Bwd Pkt Len Max    838860 non-null  int64  
 13  Bwd Pkt Len Min    838860 non-null  int64  
 14  Bwd Pkt Len Mean   838860 non-null  float64
 15  Bwd Pkt Len Std    838860 non-null  float64
 16  Fl

In [ ]:
train_main["Timestamp"].head()

,Timestamp
135924,14/02/2018 11:41:38
571728,14/02/2018 01:28:51
1026967,14/02/2018 11:25:34
14149,14/02/2018 10:40:30
418098,14/02/2018 11:10:29


In [ ]:
train_main["Timestamp"] = pd.to_datetime(train_main["Timestamp"], format = "%d/%m/%Y %H:%M:%S")
train_main["hour"] = train_main["Timestamp"].dt.hour
train_main["day"] = train_main["Timestamp"].dt.day
train_main["month"] = train_main["Timestamp"].dt.month
train_main["year"] = train_main["Timestamp"].dt.year


In [ ]:
train_main.head()

,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,...,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,hour,day,month,year
135924,21,6,2018-02-14 11:41:38,5,1,1,0,0,0,0,...,0,0.0,0.0,0,0,FTP-BruteForce,11,14,2,2018
571728,53,17,2018-02-14 01:28:51,492,1,1,46,174,46,46,...,0,0.0,0.0,0,0,Benign,1,14,2,2018
1026967,3389,6,2018-02-14 11:25:34,3467731,10,7,1148,1581,677,0,...,0,0.0,0.0,0,0,Benign,11,14,2,2018
14149,21,6,2018-02-14 10:40:30,2,1,1,0,0,0,0,...,0,0.0,0.0,0,0,FTP-BruteForce,10,14,2,2018
418098,53,17,2018-02-14 11:10:29,581,1,1,35,51,35,35,...,0,0.0,0.0,0,0,Benign,11,14,2,2018


In [ ]:
train_main.drop("Timestamp",axis=1,inplace=True)

In [ ]:
# train_main.info()

In [ ]:
from sklearn.preprocessing import OneHotEncoder
init = OneHotEncoder(sparse_output = False,handle_unknown="ignore")
Label_conversion = init.fit_transform(train_main[["Label"]]) #there are two[[]] for label because the one hot encoder takes 2d array as input

convert_dataframe = pd.DataFrame(Label_conversion,columns= init.get_feature_names_out(["Label"]))

train_main = pd.concat([train_main,convert_dataframe],axis=1)


In [ ]:
train_main.drop("Label",axis=1,inplace=True)

In [ ]:
# train_main.info()

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.compose import make_column_selector
from sklearn.base import BaseEstimator, TransformerMixin


class Timefeature(BaseEstimator, TransformerMixin):
  def fit(self,X,y=None):
    return self

  def transform(self,X):
    X = X.copy()
    X["Timestamp"] = pd.to_datetime(X["Timestamp"])
    X["hour"] = X["Timestamp"].dt.hour
    X["minute"] = X["Timestamp"].dt.minute
    X["dayofweek"] = X["Timestamp"].dt.dayofweek
    return X.drop("Timestamp", axis=1)



cat_pipeline = make_pipeline(
    SimpleImputer(strategy = "most_frequent"),
    OneHotEncoder(handle_unknown = "ignore")
)

num_pipeline = make_pipeline(
    FunctionTransformer(lambda X: np.where(np.isinf(X), np.nan, X), validate=False),
    SimpleImputer(strategy = "mean"),
)

preprocessing = ColumnTransformer([
    ("cat",cat_pipeline,make_column_selector(dtype_include = object)),
    ("num",num_pipeline,make_column_selector(dtype_include = np.number))
])

time_pipeline = make_pipeline(
    Timefeature(),
    preprocessing)



In [ ]:
X = train_strat.drop("Label",axis=1)
y = train_strat["Label"]

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import make_pipeline

le = LabelEncoder()
y = le.fit_transform(y)

tree_clf = make_pipeline(
    preprocessing,
    DecisionTreeClassifier(random_state=42)
)

tree_clf.fit(X, y)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('cat',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehotencoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x7f14c06772c0>),
                                                 ('num',
                                                  Pipeline(steps=[('functiontransformer',
                                                                   FunctionTransformer(func=<function <lambda> at 0x7f141b047e20>)),
                                                                  ('simpleimputer',
                                                                   SimpleImputer())]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x7f14c0677b00>)])),
                ('decisiontreeclassifier',
                 DecisionTreeClassifier(random_state=42))])

In [ ]:
from sklearn.model_selection import cross_val_score

accuracy_scores = cross_val_score(tree_clf, X, y, scoring="accuracy", cv=10)

print("Fold Scores:", accuracy_scores)
print("Average Accuracy:", accuracy_scores.mean())

Fold Scores: [1.         1.         1.         1.         1.         0.99998808
 1.         1.         1.         1.        ]
Average Accuracy: 0.9999988079059676


In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix

y_pred = cross_val_predict(tree_clf, X, y, cv=5)

confusion_matrix(y, y_pred)

array([[534100,      0,      1],
       [     0, 154688,      0],
       [     0,      0, 150071]])

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    534101
           1       1.00      1.00      1.00    154688
           2       1.00      1.00      1.00    150071

    accuracy                           1.00    838860
   macro avg       1.00      1.00      1.00    838860
weighted avg       1.00      1.00      1.00    838860



In [ ]:
print(X.columns)

Index(['Dst Port', 'Protocol', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts',
       'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max',
       'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std',
       'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean',
       'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean',
       'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot',
       'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min',
       'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max',
       'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags',
       'Bwd URG Flags', 'Fwd Header Len', 'Bwd Header Len', 'Fwd Pkts/s',
       'Bwd Pkts/s', 'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean',
       'Pkt Len Std', 'Pkt Len Var', 'FIN Flag Cnt', 'SYN Flag Cnt',
       'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt',
       'CWE Flag Count', 'ECE Flag Cnt', 'Down/Up Ratio', 'Pkt Size Avg',
       'Fwd Seg Siz

In [ ]:
import pandas as pd

importance = tree_clf.named_steps["decisiontreeclassifier"].feature_importances_

pd.Series(importance, index=X.columns).sort_values(ascending=False).head(10)

,0
Fwd Seg Size Min,0.648495
hour,0.340941
Dst Port,0.009915
Flow Pkts/s,0.000635
Down/Up Ratio,0.000014
Protocol,0.000000
TotLen Bwd Pkts,0.000000
Flow Duration,0.000000
Tot Bwd Pkts,0.000000
Tot Fwd Pkts,0.000000
